# Portuguese Entity Gazetteer · Dicionário de Entidades do Português

307,397 entities in Portuguese (places, people, works, products, brands…) with 747 types in 26 classes, and a gazetteer ready for search-pattern-miner.

Data: [Kaggle](https://www.kaggle.com/datasets/caiquedourado/portuguese-entity-gazetteer) · Code and method: [GitHub](https://github.com/CaiqueDourado/portuguese-entity-gazetteer). On GitHub the notebook runs on `sample/` (50 entities per class).

In [1]:
%pip install -q duckdb  # already present on most images
import glob, os
import duckdb

# Kaggle: the full files; GitHub: the samples
found = glob.glob('/kaggle/input/**/entidades.parquet', recursive=True)
if found:
    base = os.path.dirname(found[0])
    ENT = f"read_parquet('{base}/entidades.parquet')"
    TSV = f'{base}/gazetteer-pt.tsv'
    LOOK = ENT
else:
    base = '../sample' if os.path.exists('../sample') else 'sample'
    ENT = f"read_csv_auto('{base}/entidades_sample.csv')"
    TSV = f'{base}/gazetteer-pt-sample.tsv'
    docs = base.replace('sample', 'docs')
    # the samples hold no common entities, so lookups also use the README example entities
    LOOK = f"(SELECT * FROM {ENT} UNION ALL SELECT * FROM read_csv_auto('{docs}/entidades-exemplo.csv'))"
TIP = f"read_csv_auto('{base}/tipos.csv')"
q = lambda sql: duckdb.sql(sql).df()
print(base, q(f'SELECT count(*) n FROM {ENT}').n[0], 'entities')

Note: you may need to restart the kernel to use updated packages.


../sample 1299 entities


## Entities per class

In [2]:
q(f'SELECT classe, count(*) entidades FROM {ENT} GROUP BY ALL ORDER BY entidades DESC')

,classe,entidades
0,Animal,50
1,Religião e mitologia,50
2,Comida e bebida,50
3,Economia e negócios,50
4,Jogo,50
5,Local,50
6,Música,50
7,Ciência e tecnologia,50
8,Personagem,50
9,Política e direito,50


## The most common types

`tipos.csv` lists every type with its class and the source spellings merged into it (*Person*, *person* → *Pessoa*).

In [3]:
q(f'SELECT * FROM {TIP} ORDER BY num_entidades DESC LIMIT 15')

,tipo,classe,num_entidades,grafias_originais
0,Produto,Produto,70413,NaN
1,Local,Local,61714,NaN
2,Pessoa,Pessoa,49462,Person | Personalidade | person
3,Música,Música,19038,Music | music
4,Artista,Pessoa,12995,Artist | artist
5,Filme,Filme e TV,12802,FIlm | Film
6,Cidade,Local,10179,City
7,Esporte,Esporte,8129,Sport | sport
8,Empresa,Empresa e marca,6869,Company
9,Jogo,Jogo,6653,jogo


## Several types per entity

`>` separates **several types**, not a hierarchy: `Local > Cidade` is a place and a city. `classe` comes from the first type that has a class.

In [4]:
q(f"""SELECT len(string_split(tipos, ' > ')) num_tipos, count(*) entidades, any_value(entidade) exemplo, any_value(tipos) tipos
      FROM {ENT} GROUP BY ALL ORDER BY num_tipos""")

,num_tipos,entidades,exemplo,tipos
0,1,1101,abate animal,Animal
1,2,188,apogonidae,Animal > Peixe
2,3,10,antônio chimango,Livro > Pessoa > Produto


## Find the entities of a search

Whole words, longest key first, accents and case ignored: the rule search-pattern-miner uses.

In [5]:
import unicodedata

def fold(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s.lower()) if unicodedata.category(c) != 'Mn')

keys = {fold(k): (e, t) for k, e, t in duckdb.sql(f'SELECT chave, entidade, tipos FROM {LOOK}').fetchall()}
longest = max(len(k.split()) for k in keys)

def entities(search):
    words, out, i = search.split(), [], 0
    while i < len(words):
        for n in range(min(longest, len(words) - i), 0, -1):
            hit = keys.get(fold(' '.join(words[i:i + n])))
            if hit:
                out.append((' '.join(words[i:i + n]), *hit)); i += n; break
        else:
            i += 1
    return out

for s in ['hotel barato em salvador', 'como fazer pão de queijo', 'ingressos show em são paulo']:
    print(s, '->', entities(s))

hotel barato em salvador -> [('hotel', 'Hotel', 'Turismo'), ('salvador', 'Salvador', 'Local > Cidade')]
como fazer pão de queijo -> [('pão de queijo', 'pão de queijo', 'Comida')]
ingressos show em são paulo -> [('são paulo', 'São Paulo', 'Local > Cidade')]


## With search-pattern-miner

The gazetteer file is ready for [search-pattern-miner](https://github.com/CaiqueDourado/search-pattern-miner). On Kaggle, turn the internet on to install it.

In [6]:
try:
    %pip install -q "search-pattern-miner @ git+https://github.com/CaiqueDourado/search-pattern-miner#subdirectory=python"
    from search_pattern_miner import Gazetteer, mine
    gaz = Gazetteer.load(TSV) if found else Gazetteer([(k, t) for k, e, t in duckdb.sql(f'SELECT chave, entidade, tipos FROM {LOOK}').fetchall()])
    for s in ['hotel barato em salvador', 'receita de bolo de cenoura']:
        print(s, '->', mine(s, gaz, ['barbosa', 'rosa']))
except Exception as e:
    print('search-pattern-miner not available here:', type(e).__name__)

Note: you may need to restart the kernel to use updated packages.


hotel barato em salvador -> {'entity_prefix': ['[turismo] barat', '[turismo] barat [local]', '[turismo] barat [cidade]'], 'entity_skipgram': ['[turismo] [local]', '[turismo] [cidade]']}
receita de bolo de cenoura -> {'entity_prefix': ['receit [comida]'], 'entity_skipgram': []}


## Short keys

Keys of up to 3 characters (`chave_curta`) are acronyms and short names (*SP*, *AMD*, *uva*). Match them as whole words: inside other words they create false hits.

In [7]:
q(f'SELECT classe, count(*) curtas, any_value(entidade) exemplo FROM {ENT} WHERE chave_curta GROUP BY ALL ORDER BY curtas DESC LIMIT 10')

,classe,curtas,exemplo
0,Empresa e marca,4,CSD
1,Veículo,3,amv
2,Ciência e tecnologia,1,wrl
3,Economia e negócios,1,SCM
4,Jogo,1,smi
5,Educação,1,TOW
6,Religião e mitologia,1,HBS


## Limits

Common words can be entities with odd types (*brasil* is an `Animal`, *casa* is `Local > Imóvel`, *love* is a `Pessoa`): the dictionary was made to replace words inside searches. Check the type before trusting a single-word match.

In [8]:
q(f"SELECT entidade, tipos, classe FROM {LOOK} WHERE chave IN ('brasil', 'casa', 'love', 'salvador', 'hotel')")

,entidade,tipos,classe
0,Brasil,Animal,Animal
1,Casa,Local > Imóvel,Local
2,Hotel,Turismo,Local
3,Love,Pessoa,Pessoa
4,Salvador,Local > Cidade,Local
